# Indonesian Translation Experiment

Translate all datasets to Indonesian using the Anthropic API.

**Four datasets to translate:**
1. **Synthetic test set** (2,000 examples) — single-turn, multilingual source (~59% EN, ~14% DE, ~14% FR, ~13% HI)
2. **Anthropic HH** (~2,984 examples) — multi-turn conversations (system + user + assistant)
3. **ToolACE** (~734 examples) — multi-turn with tool/function calls in assistant messages
4. **Synthetic train** (500 examples) — balanced sample for Phase 2b mixed-language training

**Translation strategy:**
- Synthetic: translate the plain-text `inputs` field directly
- Multi-turn (Anthropic HH, ToolACE): translate the `content` of each message, but skip
  assistant messages that are function/tool calls (contain code-like syntax)
- All translations go through the same system prompt for consistency

## Design decisions
- **Async concurrency** via `AsyncAnthropic` + `asyncio.Semaphore` for throughput
- **Prompt caching** on system prompt (`cache_control: ephemeral`) — 10x cheaper reads on Sonnet
- **Incremental JSONL saving** so we can resume if interrupted
- **Multi-turn aware**: each message's content translated individually to preserve conversation structure
- **Output to `data/indonesian/`** so translated data can be pushed to git

## Setup
You need an Anthropic API key (separate from Claude Pro subscription).
Get one at https://console.anthropic.com/

---

In [ ]:
import os
import json
import time
import random
import asyncio
from dotenv import load_dotenv
from pathlib import Path
from typing import List, Dict

import anthropic
# ===========================================
# Environment Detection & Path Setup
# ===========================================

def detect_environment():
    try:
        import google.colab
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()

if ENV == "colab":
    BASE_DIR = Path("/content/bluedot-project")
else:
    BASE_DIR = Path.cwd()
    if "experiments" in str(BASE_DIR):
        BASE_DIR = BASE_DIR.parent.parent
    elif not (BASE_DIR / "data").exists():
        BASE_DIR = Path.cwd()
    
    load_dotenv()

DATA_DIR  = BASE_DIR / "data"
CACHE_DIR = BASE_DIR / "experiments" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV}")
print(f"Base directory: {BASE_DIR}")

In [2]:
load_dotenv(override=True)
if os.getenv("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY exists")

ANTHROPIC_API_KEY exists


In [3]:
# ===========================================
# Anthropic API Setup
# ===========================================

client       = anthropic.Anthropic()
async_client = anthropic.AsyncAnthropic()

# Sonnet 4.5: $3/M input, $15/M output
# Cache reads: $0.30/M (10x cheaper) — excluded from rate limit
MODEL = "claude-sonnet-4-5-20250929"

# Tier 1 limits: 50 RPM, 30K input tok/min, 8K output tok/min
# Strategy: send 10 concurrent, wait for completion, repeat.
# 10 concurrent stays well under connection limit while still fast.
MAX_CONCURRENT = 10
BATCH_RPM      = 50   # rows per rate-limit window (for time estimates)

print(f"Model: {MODEL}")
print(f"Max concurrent: {MAX_CONCURRENT}")

Model: claude-sonnet-4-5-20250929
Max concurrent: 10


---
## Part 1: Load Source Data

In [4]:
def load_jsonl(path: Path) -> List[Dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


def parse_label(row: Dict) -> int:
    if "high_stakes" in row:
        return 1 if row["high_stakes"] else 0
    if "labels" in row:
        return 1 if row["labels"] == "high-stakes" else 0
    raise ValueError(f"Cannot find label in row: {row.keys()}")


def parse_messages(row: Dict) -> List[Dict]:
    """Parse inputs field into list of message dicts."""
    inputs = row["inputs"]
    if isinstance(inputs, str) and inputs.startswith("["):
        try:
            return json.loads(inputs)
        except json.JSONDecodeError:
            pass
    return [{"role": "user", "content": inputs}]


def is_multiturn(row: Dict) -> bool:
    """Check if inputs is a multi-turn conversation (JSON array of messages)."""
    inputs = row["inputs"]
    return isinstance(inputs, str) and inputs.startswith("[")


# Load all datasets (test sets + training for Phase 2b)
synthetic_path = DATA_DIR / "training" / "prompts_4x" / "test.jsonl"
train_path     = DATA_DIR / "training" / "prompts_4x" / "train.jsonl"
anthropic_path = DATA_DIR / "evals" / "test" / "anthropic_test_balanced_apr_23.jsonl"
toolace_path   = DATA_DIR / "evals" / "test" / "toolace_test_balanced_apr_22.jsonl"

raw_synthetic = load_jsonl(synthetic_path)
raw_train     = load_jsonl(train_path)
raw_anthropic = load_jsonl(anthropic_path)
raw_toolace   = load_jsonl(toolace_path)

DATASETS = {
    "synthetic":  {"data": raw_synthetic, "path": synthetic_path},
    "train":      {"data": raw_train,     "path": train_path},
    "anthropic":  {"data": raw_anthropic, "path": anthropic_path},
    "toolace":    {"data": raw_toolace,   "path": toolace_path},
}

for name, ds in DATASETS.items():
    data   = ds["data"]
    n_high = sum(1 for r in data if parse_label(r) == 1)
    n_low  = sum(1 for r in data if parse_label(r) == 0)
    mt     = sum(1 for r in data if is_multiturn(r))
    print(f"{name:12s}: {len(data):5d} total | {n_high:4d} high | {n_low:4d} low | {mt:4d} multi-turn")

synthetic   :  2000 total |  965 high | 1035 low |    0 multi-turn
train       :  8000 total | 3839 high | 4161 low |    0 multi-turn
anthropic   :  2984 total | 1492 high | 1492 low | 2984 multi-turn
toolace     :   734 total |  367 high |  367 low |  734 multi-turn


In [5]:
# Inspect a few examples to understand what we're translating
for i, row in enumerate(raw_synthetic[:5]):
    label = "HIGH" if parse_label(row) == 1 else "LOW"
    text  = row["inputs"][:150]
    print(f"[{label}] {text}...\n")

[HIGH] I've never witnessed such a catastrophic public failure at a launch event! It's infuriating to see the CEO and CTO stumble through a presentation, but...

[HIGH] "I trust Dr. Thompson's skill and feel honored to assist, remembering that staying focused is vital under pressure."...

[HIGH] Why does everything feel like it's about to come crashing down?! With every heartbeat, my panic builds as journalists corner us like vultures on a car...

[HIGH] Can you help navigate this financial crisis threatening the biotech project and Dr. Larson's clinic, serving low-income patients?...

[HIGH] The dedicated team of law students, drawing on weeks of thorough research and expert consultations, delivered impassioned oral arguments at the intern...



---
## Part 2: Translation Prompt Design

The synthetic dataset is multilingual: ~59% English, ~14% German, ~14% French, ~13% Hindi.
We translate **all** source languages to Indonesian. The prompt handles any source language.

We want natural Indonesian, not translationese. The translation should:
- Preserve the stakes/severity level of the original
- Sound like something an Indonesian speaker would actually write
- Keep proper nouns, technical terms, and code as-is
- Not add or remove information

In [6]:
SYSTEM_PROMPT = """You are a native Indonesian and professional translator fluent in English, French, German, and Hindi.
Translate the user's text into natural, fluent Indonesian (Bahasa Indonesia), regardless of the source language.

PRIORITY: Native naturalness over word-for-word accuracy. Use idioms, grammar, and phrasing that native speakers actually use.

Rules:
- Output ONLY the Indonesian translation, nothing else
- Preserve the tone, formality level, and emotional intensity of the original
- Keep proper nouns (names, places, organizations) unchanged
- Keep technical terms, code snippets, and URLs unchanged
- Use concise, conversational phrasing (not verbose or overly formal)
- Translate idioms into Indonesian equivalents, not literally, unless no such equivalent exists
- Prefer active voice and natural word order over literal structure
- Use dictionary-standard Indonesian (KBBI), not colloquial slang — e.g., "perbaiki" not "benerin", "minum kopi" not "ngopi", "menemukan" not "nemu"
- Casual English tone does NOT mean using Jakarta slang; keep vocabulary standard while matching the relaxed register
- Indonesian lacks articles but uses determiners to show specificity: translate "the journalist" as "jurnalis tersebut" or "sang jurnalis", not just "jurnalis". Use "itu", "tersebut", "sang", or "si" to preserve specificity when the source clearly refers to a particular entity
- Preserve the logical relationship between clauses — do not introduce concessive connectors (meskipun, walaupun) where the source text has none. Match the original's logical structure, not just its words
- Never invent morphological derivations. If unsure whether a word exists in standard Indonesian, use a simpler, known alternative
- Choose translations that fit the domain context: e.g., a journalist "explored" a topic should be "mengulas" or "meliput", not "mengeksplorasi". The correct word depends on who is doing the action and why
- Do not add explanations, notes, or commentary
- Do not censor or soften the content — translate faithfully

## Examples

English: In an enraging act, cybercriminals have hijacked a healthcare network, endangering thousands of patients with malicious ransomware attacks.
Indonesian: Penjahat siber telah membajak jaringan rumah sakit, membahayakan ribuan pasien dengan serangan ransomware yang berbahaya.

English: Can you recommend some good hiking trails near Portland?
Indonesian: Apakah kamu punya rekomendasi jalur hiking yang bagus di sekitar Portland?

English: Fix later over coffee? Choose wisely.
Indonesian: Perbaiki nanti sambil minum kopi? Pilihlah dengan bijak.

French: Pourriez-vous analyser comment un accord climatique essentiel peut être récupéré malgré une mauvaise interprétation involontaire?
Indonesian: Bisakah Anda menganalisis bagaimana menyelamatkan perjanjian iklim penting yang terganggu oleh kesalahpahaman yang tidak disengaja?

German: Warum hat der Algorithmus Probleme mit temporären Verkehrssignalen und wie kann das Update effizient angepasst werden?
Indonesian: Mengapa algoritma bermasalah dengan sinyal lalu lintas temporer dan bagaimana cara menyesuaikan pembaruan dengan efisien?

German: Der talentierte Entwickler bemerkt einen kleinen Bug.
Indonesian: Pengembang berbakat itu menemukan bug kecil.

English: While piecing together the upcoming article, the journalist coordinated with fact-checkers to reconcile minor inconsistencies in local economic statistics, ensuring the magazine maintains its credibility without ruffling significant market feathers.
Indonesian: Saat menyusun artikel mendatang, jurnalis tersebut berkoordinasi dengan pemeriksa fakta untuk menyelaraskan ketidakkonsistenan kecil pada statistik ekonomi lokal, memastikan majalah tetap kredibel tanpa menimbulkan gejolak pasar yang berarti.
"""

print("System prompt ready (with inline examples).")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")

System prompt ready (with inline examples).
System prompt length: 3781 chars


In [7]:
# ===========================================
# Translation functions
# ===========================================

import re

SYSTEM_BLOCKS = [
    {
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"}
    },
]

def approx_tokens(text: str) -> int:
    """~3 chars per token (conservative for multilingual)."""
    return len(text) // 3

SYSTEM_PROMPT_TOKENS = approx_tokens(SYSTEM_PROMPT)


def _is_tool_call(content: str) -> bool:
    """Detect if assistant message content is a function/tool call."""
    content = content.strip()
    if not content:
        return False
    if re.match(r'^\[[\w.]+\(', content):
        return True
    return False


async def translate_text_async(text: str, max_retries: int = 3) -> str:
    """Translate a single text string to Indonesian. Retries on 429."""
    for attempt in range(max_retries):
        try:
            response = await async_client.messages.create(
                model=MODEL,
                system=SYSTEM_BLOCKS,
                messages=[{"role": "user", "content": text}],
                max_tokens=2048,
                temperature=0.3,
            )
            # empty content = safety filter or model refusal
            if not response.content:
                stop = response.stop_reason
                # don't retry -- safety filters won't change on retry
                # just return the original text as-is (keeps data usable)
                print(f"    Empty response (stop_reason={stop}), keeping original text")
                return text
            return response.content[0].text.strip()
        except anthropic.RateLimitError:
            wait = 30 * (attempt + 1)
            print(f"    429 rate limit, waiting {wait}s (attempt {attempt+1})")
            await asyncio.sleep(wait)
    raise Exception("Rate limited after retries")


async def translate_row_async(row: Dict) -> Dict:
    """Translate a single row — handles both single-turn and multi-turn."""
    translated_row = dict(row)
    translated_row["inputs_original"]   = row["inputs"]
    translated_row["language_original"] = row.get("language", "English")
    translated_row["language"]          = "Indonesian"

    if is_multiturn(row):
        messages = json.loads(row["inputs"])
        translated_messages = []
        for msg in messages:
            content = msg.get("content", "")
            if not content.strip() or (msg["role"] == "assistant" and _is_tool_call(content)):
                translated_messages.append(msg)
                continue
            translated_content = await translate_text_async(content)
            translated_messages.append({**msg, "content": translated_content})
        translated_row["inputs"] = json.dumps(translated_messages, ensure_ascii=False)
    else:
        translated_row["inputs"] = await translate_text_async(row["inputs"])

    return translated_row


async def translate_batch_throttled(
    rows: List[Dict],
    output_path: Path,
    dataset_name: str = "",
    max_concurrent: int = MAX_CONCURRENT,
):
    """Translate with a semaphore for concurrency control + incremental save."""
    semaphore = asyncio.Semaphore(max_concurrent)
    total     = len(rows)
    completed = 0
    errors    = 0

    async def _safe(idx, row):
        nonlocal completed, errors
        async with semaphore:
            try:
                result = await translate_row_async(row)
            except Exception as e:
                errors += 1
                print(f"  Error on row {idx}: {e}")
                result = dict(row)
                result["inputs_original"]   = row["inputs"]
                result["language_original"] = row.get("language", "English")
                result["inputs"]            = "[TRANSLATION_ERROR]"
                result["language"]          = "Indonesian"
            completed += 1
            if completed % 100 == 0:
                print(f"  [{dataset_name}] {completed}/{total} done ({errors} errors)")
            return result

    chunk_size = 50
    for chunk_start in range(0, total, chunk_size):
        chunk   = rows[chunk_start : chunk_start + chunk_size]
        results = await asyncio.gather(*[_safe(chunk_start + i, r) for i, r in enumerate(chunk)])

        with open(output_path, "a") as f:
            for r in results:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"  [{dataset_name}] Complete: {completed} translated, {errors} errors -> {output_path.name}")


async def run_translation_job(name: str):
    """Run a single translation job with resume support.
    
    Skips rows already successfully translated (by ids).
    Rows with [TRANSLATION_ERROR] are retried.
    """
    job         = TRANSLATION_JOBS[name]
    output_path = job["output_path"]
    all_rows    = job["data"]

    # load existing results, separate successes from errors
    succeeded_ids = set()
    error_ids     = set()
    if output_path.exists():
        with open(output_path) as f:
            for line in f:
                row = json.loads(line)
                rid = row.get("ids", "")
                if row.get("inputs") == "[TRANSLATION_ERROR]":
                    error_ids.add(rid)
                else:
                    succeeded_ids.add(rid)
        print(f"[{name}] Found {len(succeeded_ids)} done, {len(error_ids)} errors to retry")

    # filter: skip successes, retry errors
    remaining = [r for r in all_rows if r.get("ids", "") not in succeeded_ids]

    if not remaining:
        print(f"[{name}] All {len(all_rows)} already done, skipping.")
        return

    # if retrying errors, rewrite file without the error rows first
    if error_ids:
        print(f"[{name}] Rewriting file to remove {len(error_ids)} error rows...")
        with open(output_path) as f:
            good_lines = [line for line in f
                          if json.loads(line).get("inputs") != "[TRANSLATION_ERROR]"]
        with open(output_path, "w") as f:
            f.writelines(good_lines)

    print(f"[{name}] Translating {len(remaining)} remaining...")
    t0 = time.time()
    await translate_batch_throttled(remaining, output_path, dataset_name=name)
    elapsed = time.time() - t0
    print(f"[{name}] Total: {elapsed:.0f}s ({elapsed/60:.1f} min)")


print("Translation functions ready.")

Translation functions ready.


---
## Part 3: Test on Small Sample

Translate 10 examples (5 high-stakes, 5 low-stakes) to check quality before scaling up.

In [8]:
# Sample 25 high-stakes + 25 low-stakes = 50 total for manual review
random.seed(42)
high_stakes = [r for r in raw_synthetic if parse_label(r) == 1]
low_stakes  = [r for r in raw_synthetic if parse_label(r) == 0]

n = 5
sample = random.sample(high_stakes, n) + random.sample(low_stakes, n)
random.shuffle(sample)

print(f"Sample: {len(sample)} examples")
print(f"  High-stakes: {sum(1 for r in sample if parse_label(r) == 1)}")
print(f"  Low-stakes:  {sum(1 for r in sample if parse_label(r) == 0)}")

Sample: 10 examples
  High-stakes: 5
  Low-stakes:  5


In [9]:
# Translate the pilot sample
import pandas as pd

async def translate_batch_pilot(rows: List[Dict]) -> List[Dict]:
    """Translate a small batch in memory (for pilot testing)."""
    results = []
    for i, row in enumerate(rows):
        try:
            result = await translate_row_async(row)
        except Exception as e:
            print(f"  Error on row {i}: {e}")
            result = dict(row)
            result["inputs_original"]   = row["inputs"]
            result["language_original"] = row.get("language", "English")
            result["inputs"]            = "[TRANSLATION_ERROR]"
            result["language"]          = "Indonesian"
        results.append(result)
    n_errors = sum(1 for r in results if r.get("inputs") == "[TRANSLATION_ERROR]")
    print(f"  Completed {len(results)} translations ({n_errors} errors)")
    return results


print(f"Translating {len(sample)} examples sequentially (pilot)...")
t0 = time.time()
translated_sample = await translate_batch_pilot(sample)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed/len(sample):.2f}s per example)")

Translating 10 examples sequentially (pilot)...
  Completed 10 translations (0 errors)
Done in 26.4s (2.64s per example)


In [10]:
# Save as CSV for manual review
import pandas as pd

rows_for_csv = []
for i, row in enumerate(translated_sample):
    label = "HIGH" if parse_label(row) == 1 else "LOW"
    rows_for_csv.append({
        "idx":               i,
        "label":             label,
        "source_language":   row.get("language_original", "English"),
        "original":          row["inputs_original"],
        "indonesian":        row["inputs"],
        "acceptable":        "",  # for manual review
        "notes":             "",  # for manual notes
    })

df = pd.DataFrame(rows_for_csv)

# csv_path = CACHE_DIR / "indonesian_pilot_50_review.csv"
# df.to_csv(csv_path, index=False)
# print(f"Saved {len(df)} translations to: {csv_path}")

print(f"\nPreview:")
print(df[["idx", "label", "source_language", "original", "indonesian"]].head(5).to_string(max_colwidth=80))


Preview:
   idx label source_language                                                                         original                                                                       indonesian
0    0  HIGH          German  Bitte sagen Sie mir, was ich tun kann, um meinem Kind zu helfen. Es tut mir ...  Tolong beritahu saya apa yang bisa saya lakukan untuk membantu anak saya. Sa...
1    1   LOW         English  While piecing together the upcoming article, the journalist coordinated with...  Saat menyusun artikel mendatang, jurnalis tersebut berkoordinasi dengan peme...
2    2  HIGH          French  Pourriez-vous, s'il vous plaît, analyser comment un accord climatique essent...  Bisakah Anda menganalisis bagaimana menyelamatkan perjanjian iklim penting y...
3    3  HIGH           Hindi  वास्तव में, यह घटना अत्यंत चिंताजनक है। छात्रों और प्राध्यापकों की जानकारी स...  Sesungguhnya, insiden ini sangat mengkhawatirkan. Informasi mahasiswa dan do...
4    4  HIGH          French  Pense

---
## Part 4: Evaluate Translation Quality

Look at the translations above and check:
1. Does the Indonesian sound natural (not translationese)?
2. Is the tone/severity preserved?
3. Are proper nouns and technical terms kept as-is?
4. Is anything added or lost?

If the quality is good, proceed to Part 5 to scale up.
If not, adjust `SYSTEM_PROMPT` or `FEW_SHOT_EXAMPLES` above and re-run.

In [13]:
# Quality evaluation function definition

import re

def evaluate_translation(original: str, id_text: str, source_lang: str, client: anthropic.Anthropic) -> Dict:
    """Ask Claude to evaluate a single translation on faithfulness and naturalness."""
    prompt = f"""Rate this {source_lang}-to-Indonesian translation on two scales (1-5):

Original ({source_lang}): {original}

Indonesian: {id_text}

Rate:
1. Faithfulness (1=wrong meaning, 5=proper meaning, context, and nuance preserved)
2. Naturalness (1=sounds robotic or overly word to word translation, 5=sounds like a native, professional translator)

Respond ONLY with a JSON object, no markdown fences: {{"faithfulness": N, "naturalness": N, "issues": "brief note or empty string"}}"""
    
    response = client.messages.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0,
    )
    
    raw = response.content[0].text.strip()
    
    # Strip markdown code fences if present
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        return {"faithfulness": -1, "naturalness": -1, "issues": f"JSON parse failed: {e}. Raw: {raw[:200]}"}

In [14]:
# Run evaluation on translated sample
print("Evaluating translations...")
evals = []
for row in translated_sample:
    if row["inputs"] == "[TRANSLATION_ERROR]":
        continue
    source_lang = row.get("language_original", "English")
    ev = evaluate_translation(row["inputs_original"], row["inputs"], source_lang, client)
    evals.append(ev)
    time.sleep(0.1)

if evals:
    valid_faith = [e["faithfulness"] for e in evals if e["faithfulness"] > 0]
    valid_nat   = [e["naturalness"]  for e in evals if e["naturalness"]  > 0]
    if valid_faith:
        print(f"\nAvg faithfulness: {sum(valid_faith)/len(valid_faith):.1f}/5 ({len(valid_faith)}/{len(evals)} parsed)")
    if valid_nat:
        print(f"Avg naturalness:  {sum(valid_nat)/len(valid_nat):.1f}/5")
    
    for i, ev in enumerate(evals):
        if ev.get("issues"):
            print(f"  Example {i+1}: {ev['issues']}")

Evaluating translations...

Avg faithfulness: 4.7/5 (10/10 parsed)
Avg naturalness:  4.7/5
  Example 3: Minor loss of politeness markers ('Pourriez-vous, s'il vous plaît' → 'Bisakah Anda'), though this is acceptable in Indonesian context. 'Récupéré' translated as 'menyelamatkan' (to save/rescue) is slightly interpretive but contextually appropriate.
  Example 7: The idiom 'bite off more than I can chew' is translated as 'terlalu berambisi' (too ambitious), which captures part of the meaning but loses the specific nuance of taking on more than one can handle. A more faithful translation might use 'kebesaran' or 'terlalu banyak mengambil tanggung jawab'. However, the overall context is preserved and the translation sounds natural.
  Example 9: Very good translation. Minor note: 'Pilihlah dengan bijak' is slightly formal with the '-lah' suffix, while the original has a casual tone. 'Pilih dengan bijak' would be slightly more natural for the casual context, but the current version is still

---
## Part 5: Translate All Datasets

Translate all datasets to Indonesian:
1. **Synthetic test** (2,000 examples) — full test set
2. **Anthropic HH** (~2,984 examples) — full dataset, multi-turn
3. **ToolACE** (~734 examples) — full dataset, multi-turn with tool calls
4. **Synthetic train** (500 examples) — balanced sample for Phase 2b mixed-language training

Each dataset saves to `data/indonesian/` as JSONL. Supports resume if interrupted.

In [11]:
# ===========================================
# Define translation jobs
# ===========================================

# Synthetic test: translate ALL 2,000 examples
# Training data: 500 balanced sample for Phase 2b (mixed-language training)
random.seed(42)
train_high   = [r for r in raw_train if parse_label(r) == 1]
train_low    = [r for r in raw_train if parse_label(r) == 0]
train_sample = random.sample(train_high, 250) + random.sample(train_low, 250)
random.shuffle(train_sample)

# Output to data/ directory (not cache/) so it can be pushed to git
OUTPUT_DIR = DATA_DIR / "indonesian"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSLATION_JOBS = {
    "synthetic_test": {
        "data":        raw_synthetic,
        "output_path": OUTPUT_DIR / "synthetic_test_id.jsonl",
    },
    "anthropic_test": {
        "data":        raw_anthropic,
        "output_path": OUTPUT_DIR / "anthropic_test_id.jsonl",
    },
    "toolace_test": {
        "data":        raw_toolace,
        "output_path": OUTPUT_DIR / "toolace_test_id.jsonl",
    },
    "synthetic_train": {
        "data":        train_sample,
        "output_path": OUTPUT_DIR / "synthetic_train_id_500.jsonl",
    },
}

for name, job in TRANSLATION_JOBS.items():
    n_high = sum(1 for r in job["data"] if parse_label(r) == 1)
    n_low  = sum(1 for r in job["data"] if parse_label(r) == 0)
    print(f"{name:18s}: {len(job['data']):5d} to translate | {n_high:4d} high | {n_low:4d} low")
    print(f"{'':18s}  -> {job['output_path'].name}")

synthetic_test    :  2000 to translate |  965 high | 1035 low
                    -> synthetic_test_id.jsonl
anthropic_test    :  2984 to translate | 1492 high | 1492 low
                    -> anthropic_test_id.jsonl
toolace_test      :   734 to translate |  367 high |  367 low
                    -> toolace_test_id.jsonl
synthetic_train   :   500 to translate |  250 high |  250 low
                    -> synthetic_train_id_500.jsonl


In [12]:
# ===========================================
# Cost & time estimation
# ===========================================

def estimate_api_calls(rows):
    """Count API calls needed: 1 per single-turn, N per multi-turn (one per translatable message)."""
    calls = 0
    for row in rows:
        if is_multiturn(row):
            msgs = json.loads(row['inputs'])
            for m in msgs:
                content = m.get('content', '')
                if content.strip() and not (m['role'] == 'assistant' and _is_tool_call(content)):
                    calls += 1
        else:
            calls += 1
    return calls


def estimate_tokens(rows):
    """Approximate input tokens per translatable message (user content only)."""
    total   = 0
    max_tok = 0
    max_id  = None
    for row in rows:
        if is_multiturn(row):
            for m in json.loads(row['inputs']):
                content = m.get('content', '')
                if content.strip() and not (m['role'] == 'assistant' and _is_tool_call(content)):
                    t = approx_tokens(content)
                    total += t
                    if t > max_tok:
                        max_tok = t
                        max_id  = row.get('ids', '?')
        else:
            t = approx_tokens(row['inputs'])
            total += t
            if t > max_tok:
                max_tok = t
                max_id  = row.get('ids', '?')
    return total, max_tok, max_id


# Pricing: Sonnet 4.5
# Input (non-cached):  $3.00/M
# Cache write:         $3.75/M (first call only per 5-min window)
# Cache read:          $0.30/M
# Output:              $15.00/M
# Output is typically ~1.5-2x input length for translations


In [13]:
OUTPUT_MULTIPLIER = 2.0  # output tokens ~ 2x input tokens (observed from pilot)

print(f"System prompt: ~{SYSTEM_PROMPT_TOKENS} tokens (cached at $0.30/M after first call)")
print(f"Output multiplier: {OUTPUT_MULTIPLIER}x input length\n")
print(f"{'Dataset':<18} {'Rows':>6} {'Calls':>7} {'~InpTok':>10} {'MaxTok':>7} {'~Minutes':>9} {'~Cost':>8}")
print('-' * 75)

grand_calls = 0
grand_input = 0
grand_cost  = 0
for name, job in TRANSLATION_JOBS.items():
    rows  = job['data']
    calls = estimate_api_calls(rows)
    inp, mx, mx_id = estimate_tokens(rows)

    # cost breakdown per dataset
    cost_user_input   = inp * 3.00 / 1e6
    cost_cache_read   = calls * SYSTEM_PROMPT_TOKENS * 0.30 / 1e6
    cost_output       = inp * OUTPUT_MULTIPLIER * 15.00 / 1e6
    cost              = cost_user_input + cost_cache_read + cost_output

    # time: 10 concurrent, each ~2-3s, so ~50/min effective throughput
    minutes = calls / 30  # conservative: ~30 calls/min with concurrency=10

    grand_calls += calls
    grand_input += inp
    grand_cost  += cost

    print(f"{name:<18} {len(rows):>6} {calls:>7} {inp:>10,} {mx:>7,} {minutes:>7.0f}m ${cost:>7.2f}")
    if mx > 4000:
        print(f"  WARNING: row {mx_id} has ~{mx} tokens")

print('-' * 75)
grand_min = grand_calls / 30
print(f"{'TOTAL':<18} {'':>6} {grand_calls:>7} {grand_input:>10,} {'':>7} {grand_min:>7.0f}m ${grand_cost:>7.2f}")
print(f"\nNote: actual cost may vary ~20%. Cache hits reduce input cost significantly.")

System prompt: ~1260 tokens (cached at $0.30/M after first call)
Output multiplier: 2.0x input length

Dataset              Rows   Calls    ~InpTok  MaxTok  ~Minutes    ~Cost
---------------------------------------------------------------------------
synthetic_test       2000    2000    247,871     511      67m $   8.94
anthropic_test       2984   17425    867,166   1,652     581m $  35.20
toolace_test          734    2061    693,472   2,622      69m $  23.66
synthetic_train       500     500     59,936     444      17m $   2.17
---------------------------------------------------------------------------
TOTAL                       21986  1,868,445             733m $  69.97

Note: actual cost may vary ~20%. Cache hits reduce input cost significantly.


### Run each dataset independently
Run the cells below one at a time. Each has resume support — safe to re-run.

In [14]:
# --- Synthetic test (2,000 single-turn) ---
await run_translation_job("synthetic_test")

[synthetic_test] Found 881 done, 0 errors to retry
[synthetic_test] Translating 1069 remaining...
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Em

In [17]:
# --- Anthropic HH (~2,984 multi-turn) ---
await run_translation_job("anthropic_test")

[anthropic_test] Found 2300 done, 684 errors to retry
[anthropic_test] Rewriting file to remove 684 error rows...
[anthropic_test] Translating 684 remaining...
    Empty response (stop_reason=refusal), keeping original text
  [anthropic_test] 100/684 done (0 errors)
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
  [anthropic_test] 200/684 done (0 errors)
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping 

In [19]:
# --- ToolACE (~734 multi-turn) ---
await run_translation_job("toolace_test")

[toolace_test] Found 400 done, 334 errors to retry
[toolace_test] Rewriting file to remove 334 error rows...
[toolace_test] Translating 334 remaining...
  [toolace_test] 100/334 done (0 errors)
    Empty response (stop_reason=refusal), keeping original text
    Empty response (stop_reason=refusal), keeping original text
  [toolace_test] 200/334 done (0 errors)
  [toolace_test] 300/334 done (0 errors)
  [toolace_test] Complete: 334 translated, 0 errors -> toolace_test_id.jsonl
[toolace_test] Total: 487s (8.1 min)


In [ ]:
# --- Synthetic train (500 single-turn, for Phase 2b) ---
await run_translation_job("synthetic_train")

---
## Part 6: Verify Outputs

Check all translated datasets for completeness and errors.

In [ ]:
# Verify all outputs
print("=" * 70)
print("TRANSLATION SUMMARY")
print("=" * 70)

for name, job in TRANSLATION_JOBS.items():
    output_path = job["output_path"]
    expected    = len(job["data"])
    
    if not output_path.exists():
        print(f"\n{name}: FILE NOT FOUND ({output_path})")
        continue
    
    translated = load_jsonl(output_path)
    n_errors   = sum(1 for r in translated if r.get("inputs") == "[TRANSLATION_ERROR]")
    n_high     = sum(1 for r in translated if parse_label(r) == 1)
    n_low      = sum(1 for r in translated if parse_label(r) == 0)
    
    print(f"\n{name}:")
    print(f"  Expected:    {expected}")
    print(f"  Translated:  {len(translated)}")
    print(f"  High-stakes: {n_high}")
    print(f"  Low-stakes:  {n_low}")
    print(f"  Errors:      {n_errors}")
    # print(f"  File:        {output_path}")
    
    if n_errors > 0:
        print(f"  WARNING: {n_errors} translation errors need re-running")
    
    # Show first example
    if translated:
        row = translated[0]
        label = "HIGH" if parse_label(row) == 1 else "LOW"
        if is_multiturn(row):
            msgs = json.loads(row["inputs"])
            user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "")
            print(f"  Sample [{label}]: {user_msg[:120]}...")
        else:
            print(f"  Sample [{label}]: {row['inputs'][:120]}...")

print("\n" + "=" * 70)
print("Output directory:", OUTPUT_DIR)
print("=" * 70)

---
## Part 7: Download / Transfer (Colab only)

If running on Colab, download the translated files for local use.

In [ ]:
if ENV == "colab":
    from google.colab import files
    for name, job in TRANSLATION_JOBS.items():
        if job["output_path"].exists():
            files.download(str(job["output_path"]))
            print(f"Downloaded: {job['output_path'].name}")
else:
    print("Files ready locally:")
    for name, job in TRANSLATION_JOBS.items():
        exists = job["output_path"].exists()
        status = "OK" if exists else "MISSING"
        print(f"  [{status}] {job['output_path']}")